<a href="https://colab.research.google.com/github/1Bur1/Tuwaiq-classes---advanced-AI-python---final-project/blob/main/02_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Feature Engineering

This notebook transforms the cleaned housing data to make it more useful for machine learning.  
Steps: encode categories, scale numbers, create new features, and remove redundant ones.

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

# Load the cleaned data saved by notebook 01
# This skips all the cleaning steps since notebook 01 already did them
df_clean = pd.read_pickle("data/cleaned/ames_cleaned.pkl")

print(f"Loaded cleaned data: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
df_clean.head()

---
## Task 1 — One-Hot Encode Categorical Columns
Categorical columns like `Neighborhood` and `House Style` have no natural order.
We use `pd.get_dummies()` to convert each category into its own 0/1 column so the model can understand them.

In [ ]:
#task1
#• One-hot encode at least 2 categorical columns using pd.get_dummies()

# Guard: only encode if the original columns are still present
# (avoids KeyError when re-running the notebook without restarting the kernel)
cols_to_encode = [c for c in ["Neighborhood", "House Style"] if c in df_clean.columns]

if cols_to_encode:
    df_clean = pd.get_dummies(df_clean, columns=cols_to_encode,
                              prefix=["Neighborhood" if c == "Neighborhood" else "HouseStyle"
                                      for c in cols_to_encode])
    print(f"One-hot encoded: {cols_to_encode}")
else:
    print("Already encoded — skipping.")

# One-hot encoding was used for Neighborhood and House Style
# because these categorical variables have no natural order,
# and converting them into binary columns makes them usable for machine learning models.

---
## Task 2 — Ordinal Encode an Ordered Column
`Kitchen Qual` has a clear order: Poor to Fair to Typical to Good to Excellent.
We map it to numbers (0-4) so the model knows that Excellent > Good > Typical, etc.

In [ ]:
#task2
#• Ordinal encode 1 ordered column (e.g., quality: low → high)

qual_order = ["Po", "Fa", "TA", "Gd", "Ex"]

# Guard: only encode if not already done
if "Kitchen Qual" in df_clean.columns:
    df_clean["Kitchen Qual Ord"] = pd.Categorical(
        df_clean["Kitchen Qual"], categories=qual_order, ordered=True
    ).codes
    print("Ordinal encoding applied.")
else:
    print("Already encoded — skipping.")

print(df_clean["Kitchen Qual Ord"].value_counts().sort_index())

# Kitchen Qual was selected because it represents an ordered quality scale (poor to excellent),
# making ordinal encoding appropriate to preserve this natural ranking.

---
## Task 4 — Create Domain Features (Ratios)
We engineer 2 new features that combine existing columns in a meaningful way.
Safe division (`+ 1e-6`) is used to avoid dividing by zero.

In [ ]:
#task4
#Create 2 domain features: a meaningful ratio and one more of your choice.
#Use safe division to avoid dividing by zero

# Feature 1: Total bathrooms — adds up all bathroom types in the house
# (full bath counts as 1, half bath counts as 0.5 since it has no shower)
df_clean["total_bathrooms"] = (
    df_clean["Full Bath"] +
    df_clean["Half Bath"] * 0.5 +
    df_clean["Bsmt Full Bath"].fillna(0) +
    df_clean["Bsmt Half Bath"].fillna(0) * 0.5
)

# Feature 2: Rooms per area — how many rooms fit in each square foot of living space
# A high value means many small rooms; a low value means fewer but bigger rooms
df_clean["rooms_per_area"] = df_clean["TotRms AbvGrd"] / (df_clean["Gr Liv Area"] + 1e-6)

"""
1. total_bathrooms
"This feature sums all bathroom types into one number, which is easier for a model to use
than four separate bathroom columns."

2. rooms_per_area
"This feature captures how efficiently space is used — it may reflect layout quality
and housing density."
"""

---
## Task 5 — Create an Interaction Feature
An interaction feature multiplies two columns that are related.
`Overall Qual x Gr Liv Area` captures the idea that a big high-quality house is worth much more than a big low-quality one.

In [60]:
#task 5
#• Create 1 interaction feature: multiply two columns that logically go together (e.g., quality × area)

# Interaction feature: quality × area
df_clean["qual_area_interaction"] = df_clean["Overall Qual"] * df_clean["Gr Liv Area"]

#"This interaction feature combines quality and size, which may better capture overall property value than each feature individually."


---
## Task 6 — Log-Transform a Skewed Column
`SalePrice` is heavily skewed — a few very expensive houses pull the distribution to the right.
`np.log1p()` compresses those large values and makes the distribution more symmetric, which helps most ML models.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Task 6: Log-transform 1 skewed column using np.log1p() — show histogram before and after

# Show before and after side by side in one figure
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before (raw)
axes[0].hist(df_clean["SalePrice"], bins=40, color="steelblue", edgecolor="black")
axes[0].set_title("SalePrice — Before (Raw)")
axes[0].set_xlabel("Price ($)")
axes[0].set_ylabel("Number of Houses")

# Apply log transformation
df_clean["SalePrice_log"] = np.log1p(df_clean["SalePrice"])

# After (log)
axes[1].hist(df_clean["SalePrice_log"], bins=40, color="salmon", edgecolor="black")
axes[1].set_title("SalePrice — After log1p")
axes[1].set_xlabel("log(Price + 1)")
axes[1].set_ylabel("Number of Houses")

plt.suptitle("Effect of Log Transformation on SalePrice", fontsize=14)
plt.tight_layout()
plt.show()

# "Log transformation was applied to reduce skewness and make the distribution more
# symmetric, which helps many machine learning models learn better."

---
## Task 7 — Bin a Column into Groups
Instead of using the exact year a house was built, we group houses into 3 eras: **Old**, **Mid**, and **New**.
This makes the feature simpler and more meaningful for a model to use.

In [ ]:
# Task 7: Bin 1 column into meaningful groups (e.g., age groups: Old, Mid, New)

# Bin Year Built into 3 construction eras
# 1800 is used as the lower bound since the oldest house in the dataset is from 1872
bins = [1800, 1950, 2000, 2030]
labels = ["Old", "Mid", "New"]

df_clean["YearBuilt_bin"] = pd.cut(df_clean["Year Built"].dt.year, bins=bins, labels=labels)

# Show the count in each group
print(df_clean["YearBuilt_bin"].value_counts())

# "Binning was applied to group houses by construction era (Old: before 1950, Mid: 1950-2000, New: after 2000),
# making the feature more interpretable and capturing differences between old and modern properties."

---
## Task 8 — Remove Redundant Features
If two features are almost perfectly correlated (r > 0.95), they carry the same information.
We find those pairs and drop one to reduce noise and multicollinearity.

In [ ]:
#task8
#• Remove redundant features: find highly correlated pairs (r > 0.95) and drop one

# Select only numerical columns
num_df = df_clean.select_dtypes(include=['int64', 'float64', 'Int64'])

# Exclude the target column (SalePrice) and log-transformed version
exclude_cols = ["SalePrice", "SalePrice_log"]
num_df = num_df.drop(columns=[col for col in exclude_cols if col in num_df.columns])

# Compute absolute correlation matrix
corr_matrix = num_df.corr().abs()

# Keep only upper triangle to avoid duplicate pairs
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find columns with correlation > 0.95
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]

print("Columns to drop (r > 0.95):", to_drop)

# Drop redundant columns
df_clean = df_clean.drop(columns=to_drop)

print("No highly correlated pairs found — no columns dropped." if not to_drop else f"Dropped: {to_drop}")

---
## Task 3 — Scale Numerical Columns
Large columns like `Lot Area` can dominate a model just because of their big numbers.
`StandardScaler` rescales them so every feature has **mean = 0** and **std = 1**, putting them on equal footing.

In [ ]:
# Task 3: Scale at least 2 numerical columns using StandardScaler

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df_clean[["Gr Liv Area_scaled", "Lot Area_scaled"]] = scaler.fit_transform(
    df_clean[["Gr Liv Area", "Lot Area"]]
)

# Verify the scaling worked: mean should be ~0, std should be ~1
print("Gr Liv Area_scaled — mean:", round(df_clean["Gr Liv Area_scaled"].mean(), 4),
      "| std:", round(df_clean["Gr Liv Area_scaled"].std(), 4))
print("Lot Area_scaled    — mean:", round(df_clean["Lot Area_scaled"].mean(), 4),
      "| std:", round(df_clean["Lot Area_scaled"].std(), 4))

# "StandardScaler was applied so both features have mean 0 and standard deviation 1,
# preventing large-scale features from dominating the model."

---
## Task 9 — Save the Engineered Data
We save the final DataFrame (with all new features) so notebooks 03 and 04 can load it directly without repeating any of these steps.

In [ ]:
import os

# Create the folder if it does not exist
os.makedirs("data/engineered", exist_ok=True)

# Save the engineered DataFrame as a pickle file so all column types are preserved
df_clean.to_pickle("data/engineered/ames_engineered.pkl")

print("Engineered data saved to: data/engineered/ames_engineered.pkl")
print(f"Rows: {df_clean.shape[0]}, Columns: {df_clean.shape[1]}")